# Feature Engineering

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

DATA_DIR = Path("../data/raw")  # original datasets
PROCESSED_DIR = Path("../data/processed")   # processed datasets

PROCESSED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RANDOM_STATE = 21352822
N_SPLITS = 5    # 5-fold cross validation

PROTECTED_COLUMNS = {
    "SK_ID_CURR",
    "TARGET",
    "FOLD"
}

print("DATA_DIR     :", DATA_DIR.resolve())
print("PROCESSED_DIR:", PROCESSED_DIR.resolve())


DATA_DIR     : /Users/ethan/Code Repository/MSDM 5054/Project/Project 1/data/raw
PROCESSED_DIR: /Users/ethan/Code Repository/MSDM 5054/Project/Project 1/data/processed


## Basic functions Define

In [2]:
def safe_divide(a, b):
    b = b.replace(0, np.nan)
    result = a / b
    return result.replace([np.inf, -np.inf], np.nan)


def grouped_min_count_sums(df, group_cols, column_map):
    result = (
        df.groupby(group_cols, sort=False, dropna=False)[list(column_map.keys())]
        .sum(min_count=1)
        .rename(columns=column_map)
        .reset_index()
    )

    return result


def group_slope(df, group_col, x_col, y_col, output_col):
    # calculate the linear trend of y over x for each customer
    temp = df[[group_col, x_col, y_col]].dropna().copy()

    temp["xy"] = temp[x_col] * temp[y_col]
    temp["x2"] = temp[x_col] ** 2

    stats = temp.groupby(group_col).agg(
        n=(x_col, "size"),
        x_sum=(x_col, "sum"),
        y_sum=(y_col, "sum"),
        xy_sum=("xy", "sum"),
        x2_sum=("x2", "sum")
    )

    numerator = (
        stats["n"] * stats["xy_sum"]
        - stats["x_sum"] * stats["y_sum"]
    )

    denominator = (
        stats["n"] * stats["x2_sum"]
        - stats["x_sum"] ** 2
    )

    stats[output_col] = numerator / denominator
    stats[output_col] = stats[output_col].replace(
        [np.inf, -np.inf],
        np.nan
    )

    return stats[[output_col]].reset_index()


def merge_feature_table(base, features, name):
    old_cols = base.shape[1]

    result = base.merge(
        features,
        on="SK_ID_CURR",
        how="left"
    )

    print(f"{name}: +{result.shape[1] - old_cols} features")

    return result

## Load cleaned data

In [3]:
APP_TRAIN_FILE = (PROCESSED_DIR / "application_train_clean.parquet")
APP_TEST_FILE = (PROCESSED_DIR / "application_test_clean.parquet")

app_train = pd.read_parquet(APP_TRAIN_FILE)
app_test = pd.read_parquet(APP_TEST_FILE)

print("application train:", app_train.shape)
print("application test :", app_test.shape)


application train: (307511, 123)
application test : (48744, 122)


`Application` 主表
使用申请人在当前贷款申请时的信息构造new features

1. 年龄和就业年限：把以“天”为单位的负数日期转换成年。
2. 比例特征：例如贷款/收入、年金/收入、贷款/商品价格、人均收入等。
3. EXT_SOURCE 汇总：对 3 个外部评分计算均值、最小值、最大值、标准差和有效评分个数。
4. 文档数量：统计申请人有多少 FLAG_DOCUMENT_* 标记为 1。
5. 记录原始行数、客户 ID 和 TARGET，方便后面验证 merge 之后有没有破坏主表。
train 和 test 仍然保持一位客户一行，只是增加了一些基于 application 原始字段构造的新列。

In [4]:
APPLICATION_ENGINEERED_DESCRIPTIONS = {
    "APP_AGE_YEARS": 
        "Applicant age in years.",

    "APP_EMPLOYMENT_YEARS": 
        "Employment duration in years.",

    "APP_CREDIT_INCOME_RATIO":
        "Credit divided by total income.",

    "APP_ANNUITY_INCOME_RATIO":
        "Annuity divided by total income.",

    "APP_CREDIT_ANNUITY_RATIO":
        "Credit divided by annuity.",

    "APP_CREDIT_GOODS_RATIO":
        "Credit divided by goods price.",

    "APP_INCOME_PER_PERSON":
        "Income divided by family size.",

    "APP_INCOME_PER_CHILD":
        "Income divided by number of children.",

    "APP_EMPLOYED_AGE_RATIO":
        "Employment duration relative to age.",

    "APP_EXT_SOURCE_MEAN":
        "Mean of EXT_SOURCE_1, 2 and 3.",

    "APP_EXT_SOURCE_MIN":
        "Minimum EXT_SOURCE score.",

    "APP_EXT_SOURCE_MAX":
        "Maximum EXT_SOURCE score.",

    "APP_EXT_SOURCE_STD":
        "Standard deviation of EXT_SOURCE scores.",

    "APP_EXT_SOURCE_COUNT":
        "Number of available EXT_SOURCE scores.",

    "APP_DOCUMENT_COUNT":
        "Number of positive document flags."
}


def add_application_features(df):
    df = df.copy()

    df["APP_AGE_YEARS"] = (
        -df["DAYS_BIRTH"] / 365.243
    )

    df["APP_EMPLOYMENT_YEARS"] = (
        -df["DAYS_EMPLOYED"] / 365.243
    )

    df["APP_CREDIT_INCOME_RATIO"] = (
        safe_divide(
            df["AMT_CREDIT"],
            df["AMT_INCOME_TOTAL"]
        )
    )

    df["APP_ANNUITY_INCOME_RATIO"] = (
        safe_divide(
            df["AMT_ANNUITY"],
            df["AMT_INCOME_TOTAL"]
        )
    )

    df["APP_CREDIT_ANNUITY_RATIO"] = (
        safe_divide(
            df["AMT_CREDIT"],
            df["AMT_ANNUITY"]
        )
    )

    df["APP_CREDIT_GOODS_RATIO"] = (
        safe_divide(
            df["AMT_CREDIT"],
            df["AMT_GOODS_PRICE"]
        )
    )

    df["APP_INCOME_PER_PERSON"] = (
        safe_divide(
            df["AMT_INCOME_TOTAL"],
            df["CNT_FAM_MEMBERS"]
        )
    )

    df["APP_INCOME_PER_CHILD"] = (
        safe_divide(
            df["AMT_INCOME_TOTAL"],
            df["CNT_CHILDREN"]
        )
    )

    df["APP_EMPLOYED_AGE_RATIO"] = (
        safe_divide(
            df["DAYS_EMPLOYED"],
            df["DAYS_BIRTH"]
        )
    )

    ext_cols = [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]

    df["APP_EXT_SOURCE_MEAN"] = (
        df[ext_cols].mean(axis=1)
    )

    df["APP_EXT_SOURCE_MIN"] = (
        df[ext_cols].min(axis=1)
    )

    df["APP_EXT_SOURCE_MAX"] = (
        df[ext_cols].max(axis=1)
    )

    df["APP_EXT_SOURCE_STD"] = (
        df[ext_cols].std(axis=1)
    )

    df["APP_EXT_SOURCE_COUNT"] = (
        df[ext_cols]
        .notna()
        .sum(axis=1)
    )

    document_cols = [
        col for col in df.columns
        if col.startswith(
            "FLAG_DOCUMENT_"
        )
    ]

    df["APP_DOCUMENT_COUNT"] = (
        df[document_cols]
        .sum(axis=1)
    )

    df = df.replace(
        [np.inf, -np.inf],
        np.nan
    )

    return df

train = add_application_features(app_train)
test = add_application_features(app_test)


print(
    "After application features:",
    train.shape,
    test.shape
)

After application features: (307511, 138) (48744, 137)


## Bureau features

`bureau.csv` 记录客户在其他金融机构的历史信贷，一个客户可以对应多条记录。
- 只读取部分真正需要的特征列
- 计算每笔 bureau 信贷的债务/授信比。
- 把 CREDIT_ACTIVE 转成 Active、Closed、Sold、Bad debt 等 0/1 标记。
- 根据逾期金额创建“是否发生逾期”的标记。

In [5]:
BUREAU_USECOLS = [
    "SK_ID_CURR", "SK_ID_BUREAU", "CREDIT_ACTIVE", "CREDIT_TYPE",
    "DAYS_CREDIT", "CREDIT_DAY_OVERDUE", "AMT_CREDIT_MAX_OVERDUE",
    "CNT_CREDIT_PROLONG", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_LIMIT", "AMT_CREDIT_SUM_OVERDUE"
]

bureau = pd.read_csv(
    DATA_DIR / "bureau.csv",
    usecols=BUREAU_USECOLS
)

bureau["BURO_ROW_DEBT_CREDIT_RATIO"] = safe_divide(
    bureau["AMT_CREDIT_SUM_DEBT"],
    bureau["AMT_CREDIT_SUM"]
)
bureau["BURO_IS_ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype("int")
bureau["BURO_IS_CLOSED"] = (bureau["CREDIT_ACTIVE"] == "Closed").astype("int")
bureau["BURO_IS_SOLD"] = (bureau["CREDIT_ACTIVE"] == "Sold").astype("int")
bureau["BURO_IS_BAD_DEBT"] = (bureau["CREDIT_ACTIVE"] == "Bad debt").astype("int")
bureau["BURO_HAS_OVERDUE"] = (bureau["AMT_CREDIT_SUM_OVERDUE"] > 0).astype("float").mask(bureau["AMT_CREDIT_SUM_OVERDUE"].isna())

print("bureau:", bureau.shape)

bureau: (1716428, 18)


这一单元把 `bureau` 从“一个客户多行”压缩成“一个客户一行”。
包括：
- 历史贷款总数、信贷类型数量。
- Active/Closed/Sold/Bad debt 的数量和比例。
- 历史信贷金额、债务、逾期金额等的均值、最大值和总和。
- 当前仍 Active 的贷款金额和债务。
- 最近 1 年、最近 3 年的贷款数量和金额。
- 信贷金额随时间变化的趋势 BURO_CREDIT_SUM_TREND_PER_DAY。

In [6]:
bureau_g = bureau.groupby("SK_ID_CURR", sort=False)
bureau_agg = bureau_g.agg(
        BURO_LOAN_COUNT=("SK_ID_BUREAU", "count"),
        BURO_CREDIT_TYPE_NUNIQUE=("CREDIT_TYPE", "nunique"),
        BURO_ACTIVE_COUNT=("BURO_IS_ACTIVE", "sum"),
        BURO_ACTIVE_RATE=("BURO_IS_ACTIVE", "mean"),
        BURO_CLOSED_COUNT=("BURO_IS_CLOSED", "sum"),
        BURO_CLOSED_RATE=("BURO_IS_CLOSED", "mean"),
        BURO_SOLD_COUNT=("BURO_IS_SOLD", "sum"),
        BURO_SOLD_RATE=("BURO_IS_SOLD", "mean"),
        BURO_BAD_DEBT_COUNT=("BURO_IS_BAD_DEBT", "sum"),
        BURO_BAD_DEBT_RATE=("BURO_IS_BAD_DEBT", "mean"),
        BURO_DAYS_CREDIT_MEAN=("DAYS_CREDIT", "mean"),
        BURO_DAYS_CREDIT_MIN=("DAYS_CREDIT", "min"),
        BURO_DAYS_CREDIT_MAX=("DAYS_CREDIT", "max"),
        BURO_CREDIT_SUM_MEAN=("AMT_CREDIT_SUM", "mean"),
        BURO_CREDIT_SUM_MAX=("AMT_CREDIT_SUM", "max"),
        BURO_DEBT_MEAN=("AMT_CREDIT_SUM_DEBT", "mean"),
        BURO_DEBT_MAX=("AMT_CREDIT_SUM_DEBT", "max"),
        BURO_OVERDUE_MAX=("AMT_CREDIT_SUM_OVERDUE", "max"),
        BURO_CREDIT_DAY_OVERDUE_MEAN=("CREDIT_DAY_OVERDUE", "mean"),
        BURO_CREDIT_DAY_OVERDUE_MAX=("CREDIT_DAY_OVERDUE", "max"),
        BURO_MAX_OVERDUE_MEAN=("AMT_CREDIT_MAX_OVERDUE", "mean"),
        BURO_MAX_OVERDUE_MAX=("AMT_CREDIT_MAX_OVERDUE", "max"),
        BURO_HAS_OVERDUE_RATE=("BURO_HAS_OVERDUE", "mean"),
        BURO_ROW_DEBT_CREDIT_RATIO_MEAN=("BURO_ROW_DEBT_CREDIT_RATIO", "mean"),
        BURO_ROW_DEBT_CREDIT_RATIO_MAX=("BURO_ROW_DEBT_CREDIT_RATIO", "max"),
    )
bureau_sum_columns = {
    "AMT_CREDIT_SUM": "BURO_CREDIT_SUM_SUM",
    "AMT_CREDIT_SUM_DEBT": "BURO_DEBT_SUM",
    "AMT_CREDIT_SUM_LIMIT": "BURO_LIMIT_SUM",
    "AMT_CREDIT_SUM_OVERDUE": "BURO_OVERDUE_SUM",
    "CNT_CREDIT_PROLONG": "BURO_PROLONG_SUM",
}
bureau_sums = bureau_g[list(bureau_sum_columns)].sum(min_count=1).rename(
    columns=bureau_sum_columns
)
bureau_agg = bureau_agg.join(bureau_sums).reset_index()


bureau_agg["HAS_BURO"] = np.uint8(1)

bureau_agg["BURO_TOTAL_DEBT_CREDIT_RATIO"] = safe_divide(
    bureau_agg["BURO_DEBT_SUM"],
    bureau_agg["BURO_CREDIT_SUM_SUM"]
)
bureau_agg["BURO_ACTIVE_CLOSED_RATIO"] = safe_divide(
    bureau_agg["BURO_ACTIVE_COUNT"],
    bureau_agg["BURO_CLOSED_COUNT"]
)

active = bureau[bureau["CREDIT_ACTIVE"] == "Active"]

active_agg = (active.groupby("SK_ID_CURR", sort=False)[[
    "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT",
    "AMT_CREDIT_SUM_OVERDUE",
]].sum(min_count=1).rename(columns={
    "AMT_CREDIT_SUM": "BURO_ACTIVE_CREDIT_SUM",
    "AMT_CREDIT_SUM_DEBT": "BURO_ACTIVE_DEBT_SUM",
    "AMT_CREDIT_SUM_OVERDUE": "BURO_ACTIVE_OVERDUE_SUM",
}).reset_index())

active_agg["BURO_ACTIVE_DEBT_CREDIT_RATIO"] = safe_divide(
    active_agg["BURO_ACTIVE_DEBT_SUM"],
    active_agg["BURO_ACTIVE_CREDIT_SUM"]
)

bureau_agg = bureau_agg.merge(
    active_agg,
    on="SK_ID_CURR",
    how="left",
    validate="one_to_one"
)

b1 = bureau[bureau["DAYS_CREDIT"] >= -365]
b3 = bureau[bureau["DAYS_CREDIT"] >= -(365 * 3)]

b1_g = b1.groupby("SK_ID_CURR", sort=False)
b1_agg = b1_g.agg(
    BURO_1Y_LOAN_COUNT=("SK_ID_BUREAU", "count"),
)
b1_agg[["BURO_1Y_CREDIT_SUM", "BURO_1Y_DEBT_SUM"]] = (
    b1_g[["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT"]]
    .sum(min_count=1).set_axis(
        ["BURO_1Y_CREDIT_SUM", "BURO_1Y_DEBT_SUM"], axis=1
    )
)
b1_agg = b1_agg.reset_index()

b3_g = b3.groupby("SK_ID_CURR", sort=False)
b3_agg = b3_g.agg(
    BURO_3Y_LOAN_COUNT=("SK_ID_BUREAU", "count"),
)
b3_agg[["BURO_3Y_CREDIT_SUM", "BURO_3Y_DEBT_SUM"]] = (
    b3_g[["AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT"]]
    .sum(min_count=1).set_axis(
        ["BURO_3Y_CREDIT_SUM", "BURO_3Y_DEBT_SUM"], axis=1
    )
)
b3_agg = b3_agg.reset_index()

bureau_agg = (
    bureau_agg
    .merge(b1_agg, on="SK_ID_CURR", how="left", validate="one_to_one")
    .merge(b3_agg, on="SK_ID_CURR", how="left", validate="one_to_one")
)


bureau_trend = group_slope(
    bureau, "SK_ID_CURR", "DAYS_CREDIT", "AMT_CREDIT_SUM",
    "BURO_CREDIT_SUM_TREND_PER_DAY",
)
bureau_agg = bureau_agg.merge(
    bureau_trend, on="SK_ID_CURR", how="left", validate="one_to_one"
)

print("bureau_agg:", bureau_agg.shape)

bureau_agg: (305811, 45)


## Bureau-balance features

In [7]:
bb = pd.read_csv(
        DATA_DIR / "bureau_balance.csv",
        usecols=["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"]
    )



bb_status = bb["STATUS"].astype("category")
bb_missing_status = "__MISSING__"
if bb_status.isna().any():

    bb_status = bb_status.cat.add_categories([bb_missing_status]).fillna(
        bb_missing_status
    )
bb["STATUS"] = bb_status


bb_g = bb.groupby("SK_ID_BUREAU", sort=False, observed=True)
bb_by_bureau = bb_g.agg(
    BB_MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
    BB_MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
    BB_MONTHS_BALANCE_SIZE=("MONTHS_BALANCE", "size"),
)
bb_status_counts = (
    bb.groupby(
        ["SK_ID_BUREAU", "STATUS"], sort=False,
        observed=True, dropna=False
    ).size().unstack(fill_value=0)
)
bb_status_rates = bb_status_counts.div(
    bb_by_bureau["BB_MONTHS_BALANCE_SIZE"], axis=0
)
bb_status_rates.columns = [
    f"BB_STATUS_{str(status).upper()}_MEAN"
    for status in bb_status_rates.columns
]
bb_by_bureau = bb_by_bureau.join(bb_status_rates).reset_index()

bureau_link = bureau[["SK_ID_BUREAU", "SK_ID_CURR"]].drop_duplicates()

bb_customer = bb_by_bureau.merge(
    bureau_link, on="SK_ID_BUREAU", how="left",
    validate="one_to_one", indicator=True,
)
orphan_count = int((bb_customer["_merge"] == "left_only").sum())
mapped_count = int((bb_customer["_merge"] == "both").sum())
link_without_balance = int(
    (~bureau_link["SK_ID_BUREAU"].isin(bb_by_bureau["SK_ID_BUREAU"])).sum()
)
bb_mapping_audit = pd.DataFrame([{
    "bb_unique_bureau_ids": int(len(bb_by_bureau)),
    "mapped_bureau_ids": mapped_count,
    "orphan_bureau_ids": orphan_count,
    "orphan_rate": orphan_count / max(len(bb_by_bureau), 1),
    "bureau_ids_without_balance": link_without_balance,
}])

bb_customer = bb_customer.loc[
    bb_customer["_merge"] == "both"
].drop(columns="_merge")

bb_cols = [
    c for c in bb_customer.columns
    if c not in {"SK_ID_BUREAU", "SK_ID_CURR"}
]
bb_customer_agg = (
    bb_customer.groupby("SK_ID_CURR")[bb_cols]
    .mean().add_suffix("_CUSTOMER_MEAN").reset_index()
)
size_col = next(c for c in bb_cols if "MONTHS_BALANCE_SIZE" in c)
total_history = (
    bb_customer.groupby("SK_ID_CURR")[size_col]
    .agg(lambda x: x.sum(min_count=1)).rename("BB_TOTAL_MONTH_RECORDS").reset_index()
)
bb_customer_agg = bb_customer_agg.merge(
    total_history, on="SK_ID_CURR", how="left", validate="one_to_one"
)
bb_customer_agg["HAS_BB"] = np.uint8(1)

print("bb_customer_agg:", bb_customer_agg.shape)

bb_customer_agg: (134542, 14)


## Previous application features

In [8]:
PREV_USECOLS = [
    "SK_ID_CURR", "SK_ID_PREV", "NAME_CONTRACT_STATUS",
    "AMT_ANNUITY", "AMT_APPLICATION", "AMT_CREDIT",
    "AMT_DOWN_PAYMENT", "AMT_GOODS_PRICE", "DAYS_DECISION",
    "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE",
    "DAYS_TERMINATION"
]

previous = pd.read_csv(
    DATA_DIR / "previous_application.csv",
    usecols=PREV_USECOLS
)

for c in [
    "DAYS_FIRST_DRAWING", "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION", "DAYS_LAST_DUE",
    "DAYS_TERMINATION"
]:
    previous[c] = previous[c].replace(365243, np.nan)

previous["PREV_IS_APPROVED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Approved"
).astype("int")
previous["PREV_IS_REFUSED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Refused"
).astype("int")
previous["PREV_IS_CANCELED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Canceled"
).astype("int")
previous["PREV_IS_UNUSED"] = (
    previous["NAME_CONTRACT_STATUS"] == "Unused offer"
).astype("int")

previous["PREV_CREDIT_APPLICATION_RATIO"] = safe_divide(
    previous["AMT_CREDIT"], previous["AMT_APPLICATION"]
)
previous["PREV_DOWN_PAYMENT_CREDIT_RATIO"] = safe_divide(
    previous["AMT_DOWN_PAYMENT"], previous["AMT_CREDIT"]
)
previous["PREV_PLANNED_DURATION_DAYS"] = (
    previous["DAYS_LAST_DUE_1ST_VERSION"] - previous["DAYS_FIRST_DUE"]
)
previous["PREV_ACTUAL_DURATION_DAYS"] = (
    previous["DAYS_LAST_DUE"] - previous["DAYS_FIRST_DUE"]
)
previous["PREV_TERMINATION_GAP_DAYS"] = (
    previous["DAYS_TERMINATION"] - previous["DAYS_LAST_DUE"]
)
previous["PREV_DECISION_TO_FIRST_DUE_DAYS"] = (
    previous["DAYS_FIRST_DUE"] - previous["DAYS_DECISION"]
)
previous["PREV_DECISION_TO_DRAWING_DAYS"] = (
    previous["DAYS_FIRST_DRAWING"] - previous["DAYS_DECISION"]
)

previous_g = previous.groupby("SK_ID_CURR", sort=False, observed=True)
previous_agg = (
    previous_g.agg(
        PREV_COUNT=("SK_ID_PREV", "count"),
        PREV_APPROVED_COUNT=("PREV_IS_APPROVED", "sum"),
        PREV_APPROVAL_RATE=("PREV_IS_APPROVED", "mean"),
        PREV_REFUSED_COUNT=("PREV_IS_REFUSED", "sum"),
        PREV_REFUSAL_RATE=("PREV_IS_REFUSED", "mean"),
        PREV_CANCELED_COUNT=("PREV_IS_CANCELED", "sum"),
        PREV_CANCELED_RATE=("PREV_IS_CANCELED", "mean"),
        PREV_UNUSED_COUNT=("PREV_IS_UNUSED", "sum"),
        PREV_UNUSED_RATE=("PREV_IS_UNUSED", "mean"),
        PREV_APPLICATION_MEAN=("AMT_APPLICATION", "mean"),
        PREV_APPLICATION_MAX=("AMT_APPLICATION", "max"),
        PREV_CREDIT_MEAN=("AMT_CREDIT", "mean"),
        PREV_CREDIT_MAX=("AMT_CREDIT", "max"),
        PREV_ANNUITY_MEAN=("AMT_ANNUITY", "mean"),
        PREV_ANNUITY_MAX=("AMT_ANNUITY", "max"),
        PREV_DOWN_PAYMENT_MEAN=("AMT_DOWN_PAYMENT", "mean"),
        PREV_GOODS_PRICE_MEAN=("AMT_GOODS_PRICE", "mean"),
        PREV_DAYS_DECISION_MEAN=("DAYS_DECISION", "mean"),
        PREV_DAYS_DECISION_MIN=("DAYS_DECISION", "min"),
        PREV_DAYS_DECISION_MAX=("DAYS_DECISION", "max"),
        PREV_PLANNED_DURATION_MEAN=("PREV_PLANNED_DURATION_DAYS", "mean"),
        PREV_PLANNED_DURATION_MAX=("PREV_PLANNED_DURATION_DAYS", "max"),
        PREV_ACTUAL_DURATION_MEAN=("PREV_ACTUAL_DURATION_DAYS", "mean"),
        PREV_ACTUAL_DURATION_MAX=("PREV_ACTUAL_DURATION_DAYS", "max"),
        PREV_TERMINATION_GAP_MEAN=("PREV_TERMINATION_GAP_DAYS", "mean"),
        PREV_DECISION_TO_FIRST_DUE_MEAN=("PREV_DECISION_TO_FIRST_DUE_DAYS", "mean"),
        PREV_DECISION_TO_DRAWING_MEAN=("PREV_DECISION_TO_DRAWING_DAYS", "mean"),
        PREV_CREDIT_APPLICATION_RATIO_MEAN=(
            "PREV_CREDIT_APPLICATION_RATIO", "mean"
        ),
        PREV_CREDIT_APPLICATION_RATIO_MAX=(
            "PREV_CREDIT_APPLICATION_RATIO", "max"
        ),
        PREV_DOWN_PAYMENT_CREDIT_RATIO_MEAN=(
            "PREV_DOWN_PAYMENT_CREDIT_RATIO", "mean"
        ),
    )
)
previous_agg["PREV_CREDIT_SUM"] = previous_g["AMT_CREDIT"].sum(min_count=1)
previous_agg = previous_agg.reset_index()

previous_agg["HAS_PREV"] = np.uint8(1)

previous_agg["PREV_APPROVED_REFUSED_RATIO"] = safe_divide(
    previous_agg["PREV_APPROVED_COUNT"],
    previous_agg["PREV_REFUSED_COUNT"]
)

prev_1y = previous.loc[previous["DAYS_DECISION"] >= -365]
prev_3y = previous.loc[previous["DAYS_DECISION"] >= -(365 * 3)]
prev_1y_g = prev_1y.groupby("SK_ID_CURR", sort=False, observed=True)
prev_3y_g = prev_3y.groupby("SK_ID_CURR", sort=False, observed=True)
prev_1y_agg = (
    prev_1y_g.agg(
        PREV_1Y_COUNT=("SK_ID_PREV", "count"),
        PREV_1Y_APPROVAL_RATE=("PREV_IS_APPROVED", "mean"),
        PREV_1Y_REFUSAL_RATE=("PREV_IS_REFUSED", "mean"),
    )
)
prev_1y_agg["PREV_1Y_CREDIT_SUM"] = prev_1y_g["AMT_CREDIT"].sum(min_count=1)
prev_1y_agg = prev_1y_agg.reset_index()
prev_3y_agg = prev_3y_g.agg(
    PREV_3Y_COUNT=("SK_ID_PREV", "count"),
    PREV_3Y_APPROVAL_RATE=("PREV_IS_APPROVED", "mean"),
    PREV_3Y_REFUSAL_RATE=("PREV_IS_REFUSED", "mean"),
)
prev_3y_agg["PREV_3Y_CREDIT_SUM"] = prev_3y_g["AMT_CREDIT"].sum(min_count=1)
prev_3y_agg = prev_3y_agg.reset_index()
previous_agg = (previous_agg
    .merge(prev_1y_agg, on="SK_ID_CURR", how="left", validate="one_to_one")
    .merge(prev_3y_agg, on="SK_ID_CURR", how="left", validate="one_to_one")
)


previous_trend = group_slope(
    previous, "SK_ID_CURR", "DAYS_DECISION", "AMT_CREDIT",
    "PREV_CREDIT_TREND_PER_DAY",
)
previous_agg = previous_agg.merge(
    previous_trend, on="SK_ID_CURR", how="left", validate="one_to_one"
)

print("previous_agg:", previous_agg.shape)

previous_agg: (338857, 43)


## Installment payment features

In [9]:
INST_USECOLS = [
    "SK_ID_CURR", "SK_ID_PREV", "NUM_INSTALMENT_VERSION",
    "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT",
    "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"
]
INST_BASE_KEY = ["SK_ID_CURR", "SK_ID_PREV", "NUM_INSTALMENT_NUMBER"]
INST_EVENT_KEY = INST_BASE_KEY + ["DAYS_INSTALMENT"]
INST_COMPONENT_KEY = INST_EVENT_KEY + ["NUM_INSTALMENT_VERSION"]

installments = pd.read_csv(
    DATA_DIR / "installments_payments.csv", usecols=INST_USECOLS
)

raw_installment_rows = len(installments)
assert not installments[INST_COMPONENT_KEY].isna().any().any()
installments["_INST_PAYMENT_ROW_MISSING"] = (
    installments[["AMT_PAYMENT", "DAYS_ENTRY_PAYMENT"]]
    .isna().any(axis=1).astype("int")
)

component_g = installments.groupby(
    INST_COMPONENT_KEY, sort=False, dropna=False, observed=True
)
component_level = component_g.agg(
    INST_COMPONENT_RAW_ROW_COUNT=("SK_ID_CURR", "size"),
    INST_COMPONENT_PAYMENT_MISSING_ROWS=("_INST_PAYMENT_ROW_MISSING", "sum"),
    INST_COMPONENT_SCHEDULED_NUNIQUE=("AMT_INSTALMENT", "nunique"),
    INST_COMPONENT_SCHEDULED_NONMISSING=("AMT_INSTALMENT", "count"),
    AMT_INSTALMENT=("AMT_INSTALMENT", "max"),
)
component_level["INST_COMPONENT_SCHEDULED_CONFLICT"] = (
    component_level["INST_COMPONENT_SCHEDULED_NUNIQUE"] > 1
).astype("int")
component_level["INST_COMPONENT_SCHEDULE_MISSING"] = (
    component_level["INST_COMPONENT_SCHEDULED_NONMISSING"] == 0
).astype("int")
component_level = component_level.reset_index()

event_component_g = component_level.groupby(
    INST_EVENT_KEY, sort=False, dropna=False, observed=True
)
installment_level = event_component_g.agg(
    INST_COMPONENT_COUNT=("NUM_INSTALMENT_VERSION", "size"),
    INST_VERSION_NUNIQUE=("NUM_INSTALMENT_VERSION", "nunique"),
    INST_RAW_PAYMENT_ROW_COUNT=("INST_COMPONENT_RAW_ROW_COUNT", "sum"),
    INST_PAYMENT_MISSING_ROW_COUNT=("INST_COMPONENT_PAYMENT_MISSING_ROWS", "sum"),
    INST_SCHEDULED_CONFLICT_COMPONENT_COUNT=("INST_COMPONENT_SCHEDULED_CONFLICT", "sum"),
    INST_SCHEDULE_MISSING_COMPONENT_COUNT=("INST_COMPONENT_SCHEDULE_MISSING", "sum"),
)
installment_level["AMT_INSTALMENT"] = event_component_g["AMT_INSTALMENT"].sum(
    min_count=1
)
installment_level = installment_level.reset_index()


complete_payment_rows = installments["_INST_PAYMENT_ROW_MISSING"].eq(0)
payment_transactions = installments.loc[
    complete_payment_rows,
    INST_EVENT_KEY + ["DAYS_ENTRY_PAYMENT", "AMT_PAYMENT"],
].drop_duplicates()
payment_g = payment_transactions.groupby(
    INST_EVENT_KEY, sort=False, dropna=False, observed=True
)
payment_level = payment_g.agg(
    INST_UNIQUE_PAYMENT_TRANSACTION_COUNT=("AMT_PAYMENT", "size"),
    DAYS_ENTRY_PAYMENT=("DAYS_ENTRY_PAYMENT", "max"),
)
payment_level["AMT_PAYMENT"] = payment_g["AMT_PAYMENT"].sum(min_count=1)
payment_level = payment_level.reset_index()


installment_level = installment_level.merge(
    payment_level, on=INST_EVENT_KEY, how="left", validate="one_to_one"
)


installment_level["INST_UNIQUE_PAYMENT_TRANSACTION_COUNT"] = (
    installment_level["INST_UNIQUE_PAYMENT_TRANSACTION_COUNT"]
    .fillna(0).astype("int")
)
installment_level["INST_COMPLETE_PAYMENT_ROW_COUNT"] = (
    installment_level["INST_RAW_PAYMENT_ROW_COUNT"]
    - installment_level["INST_PAYMENT_MISSING_ROW_COUNT"]
)
installment_level["INST_DUPLICATE_PAYMENT_ASSOCIATION_COUNT"] = (
    installment_level["INST_COMPLETE_PAYMENT_ROW_COUNT"]
    - installment_level["INST_UNIQUE_PAYMENT_TRANSACTION_COUNT"]
).clip(lower=0).astype("int")
installment_level["INST_SPLIT_PAYMENT"] = (
    installment_level["INST_UNIQUE_PAYMENT_TRANSACTION_COUNT"] > 1
).astype("int")
installment_level["INST_PAYMENT_MISSING"] = (
    (installment_level["INST_PAYMENT_MISSING_ROW_COUNT"] > 0)
    | (installment_level["INST_UNIQUE_PAYMENT_TRANSACTION_COUNT"] == 0)
).astype("int")
installment_level["INST_CROSS_VERSION_EVENT"] = (
    installment_level["INST_VERSION_NUNIQUE"] > 1
).astype("int")
installment_level["INST_DUE_DATE_CONFLICT"] = (
    installment_level.groupby(
        INST_BASE_KEY, sort=False, dropna=False, observed=True
    )["DAYS_INSTALMENT"].transform("nunique") > 1
).astype("int")
installment_level["INST_SCHEDULED_CONFLICT"] = (
    installment_level["INST_SCHEDULED_CONFLICT_COMPONENT_COUNT"] > 0
).astype("int")
installment_level["INST_SCHEDULE_MISSING"] = (
    installment_level["INST_SCHEDULE_MISSING_COMPONENT_COUNT"] > 0
).astype("int")
installment_level.loc[
    installment_level["INST_SCHEDULE_MISSING"].eq(1), "AMT_INSTALMENT"
] = np.nan
incomplete_payment = installment_level["INST_PAYMENT_MISSING"].eq(1)
installment_level.loc[
    incomplete_payment, ["AMT_PAYMENT", "DAYS_ENTRY_PAYMENT"]
] = np.nan
installment_level["INST_DPD"] = np.maximum(
    installment_level["DAYS_ENTRY_PAYMENT"]
    - installment_level["DAYS_INSTALMENT"], 0,
)
installment_level["INST_DBD"] = np.maximum(
    installment_level["DAYS_INSTALMENT"]
    - installment_level["DAYS_ENTRY_PAYMENT"], 0,
)
installment_level["INST_LATE"] = (
    installment_level["INST_DPD"] > 0
).astype("float").mask(installment_level["INST_DPD"].isna())
installment_level["INST_PAYMENT_RATIO"] = safe_divide(
    installment_level["AMT_PAYMENT"], installment_level["AMT_INSTALMENT"]
)
installment_level["INST_PAYMENT_DIFF"] = (
    installment_level["AMT_INSTALMENT"] - installment_level["AMT_PAYMENT"]
)
installment_level["INST_UNDERPAID"] = (
    installment_level["AMT_PAYMENT"] < installment_level["AMT_INSTALMENT"]
).astype("float").mask(
    installment_level[["AMT_PAYMENT", "AMT_INSTALMENT"]].isna().any(axis=1)
)


installment_split_audit = pd.DataFrame([{
    "raw_payment_rows": int(raw_installment_rows),
    "final_event_count": int(len(installment_level)),
    "scheduled_components": int(installment_level["INST_COMPONENT_COUNT"].sum()),
    "cross_version_events": int(installment_level["INST_CROSS_VERSION_EVENT"].sum()),
    "split_payment_events": int(installment_level["INST_SPLIT_PAYMENT"].sum()),
    "missing_payment_events": int(installment_level["INST_PAYMENT_MISSING"].sum()),
    "unique_payment_transactions": int(installment_level["INST_UNIQUE_PAYMENT_TRANSACTION_COUNT"].sum()),
    "duplicate_payment_associations": int(installment_level["INST_DUPLICATE_PAYMENT_ASSOCIATION_COUNT"].sum()),
    "base_installments_with_multiple_due_dates": int(
        installment_level.loc[
            installment_level["INST_DUE_DATE_CONFLICT"].eq(1), INST_BASE_KEY
        ].drop_duplicates().shape[0]
    ),
    "scheduled_component_conflicts": int(installment_level["INST_SCHEDULED_CONFLICT_COMPONENT_COUNT"].sum()),
    "missing_scheduled_component_events": int(installment_level["INST_SCHEDULE_MISSING"].sum()),
}])

inst_customer_g = installment_level.groupby(
    "SK_ID_CURR", sort=False, observed=True
)
installments_agg = inst_customer_g.agg(
    INST_SCHEDULE_COUNT=("SK_ID_PREV", "size"),
    INST_RAW_PAYMENT_ROW_COUNT=("INST_RAW_PAYMENT_ROW_COUNT", "sum"),
    INST_COMPONENT_COUNT=("INST_COMPONENT_COUNT", "sum"),
    INST_UNIQUE_PAYMENT_TRANSACTION_COUNT=("INST_UNIQUE_PAYMENT_TRANSACTION_COUNT", "sum"),
    INST_DUPLICATE_PAYMENT_ASSOCIATION_COUNT=("INST_DUPLICATE_PAYMENT_ASSOCIATION_COUNT", "sum"),
    INST_PREV_LOAN_NUNIQUE=("SK_ID_PREV", "nunique"),
    INST_VERSION_NUNIQUE_MEAN=("INST_VERSION_NUNIQUE", "mean"),
    INST_VERSION_NUNIQUE_MAX=("INST_VERSION_NUNIQUE", "max"),
    INST_CROSS_VERSION_EVENT_RATE=("INST_CROSS_VERSION_EVENT", "mean"),
    INST_SPLIT_PAYMENT_RATE=("INST_SPLIT_PAYMENT", "mean"),
    INST_PAYMENT_MISSING_COUNT=("INST_PAYMENT_MISSING", "sum"),
    INST_PAYMENT_MISSING_RATE=("INST_PAYMENT_MISSING", "mean"),
    INST_DUE_DATE_CONFLICT_RATE=("INST_DUE_DATE_CONFLICT", "mean"),
    INST_SCHEDULED_CONFLICT_RATE=("INST_SCHEDULED_CONFLICT", "mean"),
    INST_SCHEDULE_MISSING_RATE=("INST_SCHEDULE_MISSING", "mean"),
    INST_DPD_MEAN=("INST_DPD", "mean"),
    INST_DPD_MAX=("INST_DPD", "max"),
    INST_DBD_MEAN=("INST_DBD", "mean"),
    INST_DBD_MAX=("INST_DBD", "max"),
    INST_LATE_RATE=("INST_LATE", "mean"),
    INST_UNDERPAID_RATE=("INST_UNDERPAID", "mean"),
    INST_PAYMENT_RATIO_MEAN=("INST_PAYMENT_RATIO", "mean"),
    INST_PAYMENT_RATIO_MIN=("INST_PAYMENT_RATIO", "min"),
    INST_PAYMENT_RATIO_MAX=("INST_PAYMENT_RATIO", "max"),
    INST_PAYMENT_DIFF_MEAN=("INST_PAYMENT_DIFF", "mean"),
    INST_PAYMENT_DIFF_MAX=("INST_PAYMENT_DIFF", "max"),
    INST_PAYMENT_MEAN=("AMT_PAYMENT", "mean"),
    INST_SCHEDULED_MEAN=("AMT_INSTALMENT", "mean"),
    INST_DAYS_INSTALMENT_MAX=("DAYS_INSTALMENT", "max"),
    INST_DAYS_INSTALMENT_MIN=("DAYS_INSTALMENT", "min"),
)
installments_agg["INST_DPD_SUM"] = inst_customer_g["INST_DPD"].sum(min_count=1)
installments_agg["INST_PAYMENT_DIFF_SUM"] = inst_customer_g["INST_PAYMENT_DIFF"].sum(min_count=1)
installments_agg["INST_PAYMENT_SUM"] = inst_customer_g["AMT_PAYMENT"].sum(min_count=1)
installments_agg["INST_SCHEDULED_SUM"] = inst_customer_g["AMT_INSTALMENT"].sum(min_count=1)
installments_agg = installments_agg.reset_index()

installments_agg["HAS_INST"] = np.uint8(1)


for days, label in [(180, "6M"), (365, "1Y")]:
    recent = installment_level.loc[
        installment_level["DAYS_INSTALMENT"] >= -days
    ]
    recent_agg = recent.groupby("SK_ID_CURR").agg(
        **{
            f"INST_{label}_SCHEDULE_COUNT": ("SK_ID_PREV", "size"),
            f"INST_{label}_LATE_RATE": ("INST_LATE", "mean"),
            f"INST_{label}_DPD_MEAN": ("INST_DPD", "mean"),
            f"INST_{label}_DPD_MAX": ("INST_DPD", "max"),
            f"INST_{label}_PAYMENT_RATIO_MEAN": ("INST_PAYMENT_RATIO", "mean"),
            f"INST_{label}_UNDERPAID_RATE": ("INST_UNDERPAID", "mean"),
            f"INST_{label}_MISSING_PAYMENT_RATE": ("INST_PAYMENT_MISSING", "mean"),
        }
    ).reset_index()
    installments_agg = installments_agg.merge(
        recent_agg, on="SK_ID_CURR", how="left", validate="one_to_one"
    )
    

inst_trend = group_slope(
    installment_level, "SK_ID_CURR", "DAYS_INSTALMENT",
    "INST_PAYMENT_RATIO", "INST_PAYMENT_RATIO_TREND_PER_DAY",
)
installments_agg = installments_agg.merge(
    inst_trend, on="SK_ID_CURR", how="left", validate="one_to_one"
)

print("installments_agg:", installments_agg.shape)

installments_agg: (339587, 51)


## Credit card features

In [10]:
CC_USECOLS = [
    "SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE",
    "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL",
    "AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_CURRENT",
    "AMT_RECEIVABLE_PRINCIPAL", "SK_DPD", "SK_DPD_DEF"
]
credit_card = pd.read_csv(
    DATA_DIR / "credit_card_balance.csv", usecols=CC_USECOLS
)

credit_card["CC_UTILIZATION"] = safe_divide(
    credit_card["AMT_BALANCE"], credit_card["AMT_CREDIT_LIMIT_ACTUAL"]
)
credit_card["CC_HAS_DPD"] = (credit_card["SK_DPD"] > 0).astype("int")
credit_card["CC_HAS_DPD_DEF"] = (credit_card["SK_DPD_DEF"] > 0).astype("int")

credit_card_agg = credit_card.groupby("SK_ID_CURR").agg(
    CC_RECORD_COUNT=("SK_ID_PREV", "size"),
    CC_PREV_CARD_NUNIQUE=("SK_ID_PREV", "nunique"),
    CC_MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
    CC_MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
    CC_BALANCE_MEAN=("AMT_BALANCE", "mean"),
    CC_BALANCE_MAX=("AMT_BALANCE", "max"),
    CC_LIMIT_MEAN=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    CC_LIMIT_MAX=("AMT_CREDIT_LIMIT_ACTUAL", "max"),
    CC_UTILIZATION_MEAN=("CC_UTILIZATION", "mean"),
    CC_UTILIZATION_MAX=("CC_UTILIZATION", "max"),
    CC_DRAWINGS_MEAN=("AMT_DRAWINGS_CURRENT", "mean"),
    CC_DRAWINGS_MAX=("AMT_DRAWINGS_CURRENT", "max"),
    CC_PAYMENT_MEAN=("AMT_PAYMENT_CURRENT", "mean"),
    CC_PAYMENT_MAX=("AMT_PAYMENT_CURRENT", "max"),
    CC_RECEIVABLE_MEAN=("AMT_RECEIVABLE_PRINCIPAL", "mean"),
    CC_RECEIVABLE_MAX=("AMT_RECEIVABLE_PRINCIPAL", "max"),
    CC_DPD_MEAN=("SK_DPD", "mean"),
    CC_DPD_MAX=("SK_DPD", "max"),
    CC_DPD_DEF_MEAN=("SK_DPD_DEF", "mean"),
    CC_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    CC_DPD_POSITIVE_RATE=("CC_HAS_DPD", "mean"),
    CC_DPD_DEF_POSITIVE_RATE=("CC_HAS_DPD_DEF", "mean"),
).reset_index()
cc_customer_flow_sums = grouped_min_count_sums(
    credit_card, "SK_ID_CURR", {
        "AMT_DRAWINGS_CURRENT": "CC_DRAWINGS_SUM",
        "AMT_PAYMENT_CURRENT": "CC_PAYMENT_SUM",
    }
)
credit_card_agg = credit_card_agg.merge(
    cc_customer_flow_sums, on="SK_ID_CURR", how="left", validate="one_to_one"
)

credit_card_agg["HAS_CC"] = np.uint8(1)


cc_contract_keys = ["SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE"]
cc_contract_g = credit_card.groupby(
    cc_contract_keys, sort=False, dropna=False
)
cc_contract_month = cc_contract_g.agg(
    CC_MONTH_ROW_COUNT=("SK_ID_PREV", "size"),
    AMT_BALANCE=("AMT_BALANCE", "mean"),
    AMT_CREDIT_LIMIT_ACTUAL=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    AMT_RECEIVABLE_PRINCIPAL=("AMT_RECEIVABLE_PRINCIPAL", "mean"),
    SK_DPD=("SK_DPD", "max"),
    SK_DPD_DEF=("SK_DPD_DEF", "max"),
)
cc_contract_month[["AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_CURRENT"]] = (
    cc_contract_g[["AMT_DRAWINGS_CURRENT", "AMT_PAYMENT_CURRENT"]]
    .sum(min_count=1)
)
cc_contract_month = cc_contract_month.reset_index()

cc_contract_month["CC_UTILIZATION"] = safe_divide(
    cc_contract_month["AMT_BALANCE"],
    cc_contract_month["AMT_CREDIT_LIMIT_ACTUAL"],
)


assert cc_contract_month.groupby("SK_ID_PREV")["SK_ID_CURR"].nunique().max() == 1
cc_contract_latest = (
    cc_contract_month.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"])
    .drop_duplicates("SK_ID_PREV", keep="last")
)
cc_latest_customer = cc_contract_latest.groupby("SK_ID_CURR").agg(
    CC_LATEST_CONTRACT_COUNT=("SK_ID_PREV", "nunique"),
    CC_LATEST_MONTH_MIN=("MONTHS_BALANCE", "min"),
    CC_LATEST_MONTH_MAX=("MONTHS_BALANCE", "max"),
    CC_LATEST_BALANCE_MEAN=("AMT_BALANCE", "mean"),
    CC_LATEST_BALANCE_MAX=("AMT_BALANCE", "max"),
    CC_LATEST_LIMIT_MEAN=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
    CC_LATEST_UTILIZATION_MEAN=("CC_UTILIZATION", "mean"),
    CC_LATEST_UTILIZATION_MAX=("CC_UTILIZATION", "max"),
    CC_LATEST_DPD_MEAN=("SK_DPD", "mean"),
    CC_LATEST_DPD_MAX=("SK_DPD", "max"),
    CC_LATEST_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
).reset_index()
cc_latest_sums = grouped_min_count_sums(
    cc_contract_latest, "SK_ID_CURR", {
        "AMT_BALANCE": "CC_LATEST_BALANCE_SUM",
        "AMT_CREDIT_LIMIT_ACTUAL": "CC_LATEST_LIMIT_SUM",
    }
)
cc_latest_customer = cc_latest_customer.merge(
    cc_latest_sums, on="SK_ID_CURR", how="left", validate="one_to_one"
)

cc_latest_customer["CC_LATEST_TOTAL_UTILIZATION"] = safe_divide(
    cc_latest_customer["CC_LATEST_BALANCE_SUM"],
    cc_latest_customer["CC_LATEST_LIMIT_SUM"],
)
credit_card_agg = credit_card_agg.merge(
    cc_latest_customer, on="SK_ID_CURR", how="left", validate="one_to_one"
)

cc_customer_month_g = cc_contract_month.groupby(
    ["SK_ID_CURR", "MONTHS_BALANCE"], sort=False, dropna=False
)
cc_customer_month = cc_customer_month_g.agg(
    CC_CUSTOMER_MONTH_DPD_MAX=("SK_DPD", "max"),
)
cc_customer_month[[
    "CC_CUSTOMER_MONTH_BALANCE", "CC_CUSTOMER_MONTH_LIMIT"
]] = cc_customer_month_g[["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL"]].sum(
    min_count=1
).set_axis(["CC_CUSTOMER_MONTH_BALANCE", "CC_CUSTOMER_MONTH_LIMIT"], axis=1)
cc_customer_month = cc_customer_month.reset_index()

cc_customer_month["CC_CUSTOMER_MONTH_UTILIZATION"] = safe_divide(
    cc_customer_month["CC_CUSTOMER_MONTH_BALANCE"],
    cc_customer_month["CC_CUSTOMER_MONTH_LIMIT"],
)

cc_12m = cc_customer_month.loc[cc_customer_month["MONTHS_BALANCE"] >= -12]
cc_12m_agg = cc_12m.groupby("SK_ID_CURR").agg(
    CC_12M_MONTH_COUNT=("MONTHS_BALANCE", "nunique"),
    CC_12M_BALANCE_MEAN=("CC_CUSTOMER_MONTH_BALANCE", "mean"),
    CC_12M_UTILIZATION_MEAN=("CC_CUSTOMER_MONTH_UTILIZATION", "mean"),
    CC_12M_DPD_MAX=("CC_CUSTOMER_MONTH_DPD_MAX", "max"),
).reset_index()
credit_card_agg = credit_card_agg.merge(
    cc_12m_agg, on="SK_ID_CURR", how="left", validate="one_to_one"
)

cc_trend = group_slope(
    cc_customer_month, "SK_ID_CURR", "MONTHS_BALANCE",
    "CC_CUSTOMER_MONTH_BALANCE", "CC_BALANCE_TREND_PER_MONTH",
)
credit_card_agg = credit_card_agg.merge(
    cc_trend, on="SK_ID_CURR", how="left", validate="one_to_one"
)

print("credit_card_agg:", credit_card_agg.shape)

credit_card_agg: (103558, 45)


## POS/CASH balance features

In [11]:
POS_USECOLS = [
    "SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE",
    "CNT_INSTALMENT", "CNT_INSTALMENT_FUTURE",
    "NAME_CONTRACT_STATUS", "SK_DPD", "SK_DPD_DEF"
]
pos = pd.read_csv(
    DATA_DIR / "POS_CASH_balance.csv", usecols=POS_USECOLS
)

for status_name, status_value in {
    "COMPLETED": "Completed", "ACTIVE": "Active",
    "SIGNED": "Signed", "RETURNED": "Returned to the store",
}.items():
    pos[f"POS_IS_{status_name}"] = (
        pos["NAME_CONTRACT_STATUS"] == status_value
    ).astype("int8")

pos_agg = pos.groupby("SK_ID_CURR").agg(
    POS_RECORD_COUNT=("SK_ID_PREV", "size"),
    POS_PREV_LOAN_NUNIQUE=("SK_ID_PREV", "nunique"),
    POS_MONTHS_BALANCE_MIN=("MONTHS_BALANCE", "min"),
    POS_MONTHS_BALANCE_MAX=("MONTHS_BALANCE", "max"),
    POS_INSTALMENT_MEAN=("CNT_INSTALMENT", "mean"),
    POS_INSTALMENT_MAX=("CNT_INSTALMENT", "max"),
    POS_FUTURE_INSTALMENT_MEAN=("CNT_INSTALMENT_FUTURE", "mean"),
    POS_FUTURE_INSTALMENT_MIN=("CNT_INSTALMENT_FUTURE", "min"),
    POS_DPD_MEAN=("SK_DPD", "mean"),
    POS_DPD_MAX=("SK_DPD", "max"),
    POS_DPD_DEF_MEAN=("SK_DPD_DEF", "mean"),
    POS_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    POS_COMPLETED_RATE=("POS_IS_COMPLETED", "mean"),
    POS_ACTIVE_RATE=("POS_IS_ACTIVE", "mean"),
    POS_SIGNED_RATE=("POS_IS_SIGNED", "mean"),
    POS_RETURNED_RATE=("POS_IS_RETURNED", "mean"),
).reset_index()
pos_agg["HAS_POS"] = np.uint8(1)


pos_contract_month = pos.groupby(
    ["SK_ID_CURR", "SK_ID_PREV", "MONTHS_BALANCE"], sort=False
).agg(
    POS_MONTH_ROW_COUNT=("SK_ID_PREV", "size"),
    CNT_INSTALMENT=("CNT_INSTALMENT", "max"),
    CNT_INSTALMENT_FUTURE=("CNT_INSTALMENT_FUTURE", "min"),
    SK_DPD=("SK_DPD", "max"),
    SK_DPD_DEF=("SK_DPD_DEF", "max"),
    POS_IS_COMPLETED=("POS_IS_COMPLETED", "max"),
    POS_IS_ACTIVE=("POS_IS_ACTIVE", "max"),
).reset_index()


assert pos_contract_month.groupby("SK_ID_PREV")["SK_ID_CURR"].nunique().max() == 1
pos_contract_latest = (
    pos_contract_month.sort_values(["SK_ID_PREV", "MONTHS_BALANCE"])
    .drop_duplicates("SK_ID_PREV", keep="last")
)
pos_latest_customer = pos_contract_latest.groupby("SK_ID_CURR").agg(
    POS_LATEST_CONTRACT_COUNT=("SK_ID_PREV", "nunique"),
    POS_LATEST_MONTH_MIN=("MONTHS_BALANCE", "min"),
    POS_LATEST_MONTH_MAX=("MONTHS_BALANCE", "max"),
    POS_LATEST_INSTALMENT_MEAN=("CNT_INSTALMENT", "mean"),
    POS_LATEST_FUTURE_INSTALMENT_MEAN=("CNT_INSTALMENT_FUTURE", "mean"),
    POS_LATEST_FUTURE_INSTALMENT_MAX=("CNT_INSTALMENT_FUTURE", "max"),
    POS_LATEST_DPD_MEAN=("SK_DPD", "mean"),
    POS_LATEST_DPD_MAX=("SK_DPD", "max"),
    POS_LATEST_DPD_DEF_MAX=("SK_DPD_DEF", "max"),
    POS_LATEST_COMPLETED_RATE=("POS_IS_COMPLETED", "mean"),
    POS_LATEST_ACTIVE_RATE=("POS_IS_ACTIVE", "mean"),
).reset_index()
pos_latest_sums = grouped_min_count_sums(
    pos_contract_latest, "SK_ID_CURR", {
        "CNT_INSTALMENT": "POS_LATEST_INSTALMENT_SUM",
        "CNT_INSTALMENT_FUTURE": "POS_LATEST_FUTURE_INSTALMENT_SUM",
    }
)
pos_latest_customer = pos_latest_customer.merge(
    pos_latest_sums, on="SK_ID_CURR", how="left", validate="one_to_one"
)

pos_agg = pos_agg.merge(
    pos_latest_customer, on="SK_ID_CURR", how="left", validate="one_to_one"
)

pos_customer_month_g = pos_contract_month.groupby(
    ["SK_ID_CURR", "MONTHS_BALANCE"], sort=False, dropna=False
)
pos_customer_month = pos_customer_month_g.agg(
    POS_CUSTOMER_MONTH_DPD_MAX=("SK_DPD", "max"),
    POS_CUSTOMER_MONTH_ACTIVE_RATE=("POS_IS_ACTIVE", "mean"),
)
pos_customer_month["POS_CUSTOMER_MONTH_FUTURE_SUM"] = (
    pos_customer_month_g["CNT_INSTALMENT_FUTURE"].sum(min_count=1)
)
pos_customer_month = pos_customer_month.reset_index()

pos_applicant_latest = (
    pos_customer_month.sort_values(["SK_ID_CURR", "MONTHS_BALANCE"])
    .drop_duplicates("SK_ID_CURR", keep="last")
    [["SK_ID_CURR", "MONTHS_BALANCE",
      "POS_CUSTOMER_MONTH_FUTURE_SUM", "POS_CUSTOMER_MONTH_DPD_MAX",
      "POS_CUSTOMER_MONTH_ACTIVE_RATE"]]
    .rename(columns={
        "MONTHS_BALANCE": "POS_APPLICANT_LATEST_MONTH",
        "POS_CUSTOMER_MONTH_FUTURE_SUM": "POS_APPLICANT_LATEST_FUTURE_SUM",
        "POS_CUSTOMER_MONTH_DPD_MAX": "POS_APPLICANT_LATEST_DPD_MAX",
        "POS_CUSTOMER_MONTH_ACTIVE_RATE": "POS_APPLICANT_LATEST_ACTIVE_RATE",
    })
)
assert pos_applicant_latest["SK_ID_CURR"].is_unique
pos_agg = pos_agg.merge(
    pos_applicant_latest, on="SK_ID_CURR", how="left",
    validate="one_to_one"
)


pos_12m = pos_customer_month.loc[pos_customer_month["MONTHS_BALANCE"] >= -12]
pos_12m_agg = pos_12m.groupby("SK_ID_CURR").agg(
    POS_12M_MONTH_COUNT=("MONTHS_BALANCE", "nunique"),
    POS_12M_FUTURE_INSTALMENT_MEAN=("POS_CUSTOMER_MONTH_FUTURE_SUM", "mean"),
    POS_12M_DPD_MAX=("POS_CUSTOMER_MONTH_DPD_MAX", "max"),
    POS_12M_ACTIVE_RATE=("POS_CUSTOMER_MONTH_ACTIVE_RATE", "mean"),
).reset_index()
pos_agg = pos_agg.merge(
    pos_12m_agg, on="SK_ID_CURR", how="left", validate="one_to_one"
)

pos_trend = group_slope(
    pos_customer_month, "SK_ID_CURR", "MONTHS_BALANCE",
    "POS_CUSTOMER_MONTH_FUTURE_SUM",
    "POS_FUTURE_INSTALMENT_TREND_PER_MONTH",
)
pos_agg = pos_agg.merge(
    pos_trend, on="SK_ID_CURR", how="left", validate="one_to_one"
)


print("pos_agg:", pos_agg.shape)

pos_agg: (337252, 40)


## Merge all application features tables

In [12]:
train = merge_feature_table(
    train,
    bureau_agg,
    "bureau"
)

test = merge_feature_table(
    test,
    bureau_agg,
    "bureau"
)



if bb_customer_agg is not None:
    train = merge_feature_table(
        train,
        bb_customer_agg,
        "bureau_balance"
    )

    test = merge_feature_table(
        test,
        bb_customer_agg,
        "bureau_balance"
    )

    


train = merge_feature_table(
    train,
    previous_agg,
    "previous_application"
)

test = merge_feature_table(
    test,
    previous_agg,
    "previous_application"
)




train = merge_feature_table(
    train,
    installments_agg,
    "installments"
)

test = merge_feature_table(
    test,
    installments_agg,
    "installments"
)




train = merge_feature_table(
    train,
    credit_card_agg,
    "credit_card"
)

test = merge_feature_table(
    test,
    credit_card_agg,
    "credit_card"
)




train = merge_feature_table(
    train,
    pos_agg,
    "POS_CASH"
)

test = merge_feature_table(
    test,
    pos_agg,
    "POS_CASH"
)




history_flags = [
    col for col in train.columns
    if col.startswith("HAS_")
    and col in test.columns
]

for col in history_flags:
    train[col] = (
        train[col]
        .fillna(0)
        .astype(int)
    )

    test[col] = (
        test[col]
        .fillna(0)
        .astype(int)
    )



print("Merged train:",train.shape)
print("Merged test :",test.shape)
print("History flags:",history_flags)

bureau: +44 features
bureau: +44 features
bureau_balance: +13 features
bureau_balance: +13 features
previous_application: +42 features
previous_application: +42 features
installments: +50 features
installments: +50 features
credit_card: +44 features
credit_card: +44 features
POS_CASH: +39 features
POS_CASH: +39 features
Merged train: (307511, 370)
Merged test : (48744, 369)
History flags: ['HAS_BURO', 'HAS_BB', 'HAS_PREV', 'HAS_INST', 'HAS_CC', 'HAS_POS']


## Cross source features

In [13]:
if {"BURO_DEBT_SUM", "AMT_INCOME_TOTAL"}.issubset(train.columns):
    train["CROSS_BURO_DEBT_INCOME_RATIO"] = safe_divide(
        train["BURO_DEBT_SUM"], train["AMT_INCOME_TOTAL"]
    )
    test["CROSS_BURO_DEBT_INCOME_RATIO"] = safe_divide(
        test["BURO_DEBT_SUM"], test["AMT_INCOME_TOTAL"]
    )

if {"BURO_CREDIT_SUM_SUM", "AMT_INCOME_TOTAL"}.issubset(train.columns):
    train["CROSS_BURO_CREDIT_INCOME_RATIO"] = safe_divide(
        train["BURO_CREDIT_SUM_SUM"], train["AMT_INCOME_TOTAL"]
    )
    test["CROSS_BURO_CREDIT_INCOME_RATIO"] = safe_divide(
        test["BURO_CREDIT_SUM_SUM"], test["AMT_INCOME_TOTAL"]
    )

if {"PREV_COUNT", "BURO_LOAN_COUNT"}.issubset(train.columns):
    train["CROSS_PREV_BURO_COUNT_RATIO"] = safe_divide(
        train["PREV_COUNT"], train["BURO_LOAN_COUNT"]
    )
    test["CROSS_PREV_BURO_COUNT_RATIO"] = safe_divide(
        test["PREV_COUNT"], test["BURO_LOAN_COUNT"]
    )

train.replace([np.inf, -np.inf], np.nan, inplace=True)
test.replace([np.inf, -np.inf], np.nan, inplace=True)

print("Cross-source features added.")

/var/folders/zp/04d6g4b55j72tsnry5x14nsc0000gn/T/ipykernel_44209/2370233205.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train["CROSS_BURO_DEBT_INCOME_RATIO"] = safe_divide(
/var/folders/zp/04d6g4b55j72tsnry5x14nsc0000gn/T/ipykernel_44209/2370233205.py:5: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  test["CROSS_BURO_DEBT_INCOME_RATIO"] = safe_divide(
/var/folders/zp/04d6g4b55j72tsnry5x14nsc0000gn/T/ipykernel_44209/2370233205.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling

Cross-source features added.


## Historical coverage and train&test comparison

In [14]:
history_coverage_rows = []

for flag in history_flags:
    history_coverage_rows.append({
        "history_flag":
            flag,

        "train_coverage":
            train[flag].mean(),

        "test_coverage":
            test[flag].mean(),

        "test_minus_train":
            test[flag].mean()
            - train[flag].mean()
    })

history_coverage_summary = pd.DataFrame(
    history_coverage_rows
)


numeric_features = [
    col
    for col in train.select_dtypes(
        include=np.number
    ).columns
    if col in test.columns
    and col not in PROTECTED_COLUMNS
]

numeric_drift_rows = []

for col in numeric_features:
    train_mean = train[col].mean()
    test_mean = test[col].mean()

    pooled_sd = np.sqrt(
        (
            train[col].var()
            + test[col].var()
        ) / 2
    )

    if (
        pd.isna(pooled_sd)
        or pooled_sd == 0
    ):
        smd = np.nan
    else:
        smd = (
            test_mean - train_mean
        ) / pooled_sd

    numeric_drift_rows.append({
        "feature": col,
        "train_mean": train_mean,
        "test_mean": test_mean,
        "standardized_mean_difference":
            smd,
        "train_missing_rate":
            train[col].isna().mean(),
        "test_missing_rate":
            test[col].isna().mean()
    })

numeric_drift_summary = pd.DataFrame(
    numeric_drift_rows
)

numeric_drift_summary["abs_smd"] = (
    numeric_drift_summary[
        "standardized_mean_difference"
    ].abs()
)

numeric_drift_summary = (
    numeric_drift_summary
    .sort_values(
        "abs_smd",
        ascending=False
    )
)

display(history_coverage_summary)
display(numeric_drift_summary.head(30))


,history_flag,train_coverage,test_coverage,test_minus_train
0,HAS_BURO,0.8569,0.8682,0.0114
1,HAS_BB,0.2999,0.8680,0.5681
2,HAS_PREV,0.9465,0.9806,0.0341
3,HAS_INST,0.9484,0.9836,0.0352
4,HAS_CC,0.2826,0.3416,0.0590
5,HAS_POS,0.9412,0.9808,0.0396


,feature,train_mean,test_mean,standardized_mean_difference,train_missing_rate,test_missing_rate,abs_smd
176,HAS_BB,0.2999,0.8680,1.4103,0.0000,0.0000,1.4103
165,BB_MONTHS_BALANCE_MAX_CUSTOMER_MEAN,-7.7462,-0.0310,1.0464,0.7001,0.1320,1.0464
109,APP_CREDIT_ANNUITY_RATIO,21.6123,17.1739,-0.6252,0.0000,0.0005,0.6252
166,BB_MONTHS_BALANCE_SIZE_CUSTOMER_MEAN,27.4772,36.3044,0.5172,0.7001,0.1320,0.5172
101,AMT_REQ_CREDIT_BUREAU_MON,0.2674,0.0093,-0.3956,0.1350,0.1241,0.3956
102,AMT_REQ_CREDIT_BUREAU_QRT,0.2655,0.5469,0.3776,0.1350,0.1241,0.3776
175,BB_TOTAL_MONTH_RECORDS,159.3999,224.0110,0.3493,0.7001,0.1320,0.3493
16,FLAG_EMAIL,0.0567,0.1626,0.3439,0.0000,0.0000,0.3439
107,APP_CREDIT_INCOME_RATIO,3.9576,3.1675,-0.3252,0.0000,0.0000,0.3252
168,BB_STATUS_0_MEAN_CUSTOMER_MEAN,0.4571,0.3919,-0.2734,0.7001,0.1320,0.2734


## cleanup and constant column removal

In [15]:
constant_cols = [
    col
    for col in train.columns
    if col not in PROTECTED_COLUMNS
    and train[col].nunique(
        dropna=False
    ) <= 1
]

if constant_cols:
    train = train.drop(
        columns=constant_cols
    )

    test = test.drop(
        columns=[
            col
            for col in constant_cols
            if col in test.columns
        ]
    )

print(
    "After constant removal:",
    train.shape,
    test.shape
)


After constant removal: (307511, 373) (48744, 372)


## Create shared 5-fold

In [16]:
train["FOLD"] = -1

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold, (_, valid_idx) in enumerate(
    skf.split(
        train,
        train["TARGET"]
    )
):
    train.loc[
        train.index[valid_idx],
        "FOLD"
    ] = fold

train["FOLD"] = train["FOLD"].astype(
    int
)

fold_summary = (
    train.groupby("FOLD")["TARGET"]
    .agg(
        count="size",
        target_rate="mean"
    )
)

fold_summary


,count,target_rate
FOLD,,
0,61503,0.0807
1,61502,0.0807
2,61502,0.0807
3,61502,0.0807
4,61502,0.0807


In [17]:
clean_output_cols = [
    col
    for col in train.columns
    if col not in {
        "TARGET",
        "FOLD"
    }
]

test = test[clean_output_cols].copy()


## One-hot encoded

In [18]:
def one_hot_encode_train_test(
    clean_train,
    clean_test
):
    train_meta = clean_train[
        ["SK_ID_CURR", "TARGET", "FOLD"]
    ].copy()

    test_ids = clean_test[
        ["SK_ID_CURR"]
    ].copy()

    train_features = clean_train.drop(
        columns=[
            "SK_ID_CURR",
            "TARGET",
            "FOLD"
        ]
    )

    test_features = clean_test.drop(
        columns=[
            "SK_ID_CURR"
        ]
    )

    categorical_cols = (
        train_features
        .select_dtypes(
            include=[
                "object",
                "string",
                "category"
            ]
        )
        .columns
        .tolist()
    )

    encoding_rows = []

    for col in categorical_cols:
        train_levels = set(
            train_features[col]
            .dropna()
            .unique()
        )

        test_levels = set(
            test_features[col]
            .dropna()
            .unique()
        )

        unseen = (
            test_levels
            - train_levels
        )

        encoding_rows.append({
            "feature": col,
            "train_levels":
                len(train_levels),
            "test_unseen_levels":
                len(unseen),
            "test_unseen_rows":
                test_features[col]
                .isin(unseen)
                .sum()
        })

    train_encoded_features = (
        pd.get_dummies(
            train_features,
            columns=categorical_cols,
            dummy_na=True,
            dtype=int
        )
    )

    test_encoded_features = (
        pd.get_dummies(
            test_features,
            columns=categorical_cols,
            dummy_na=True,
            dtype=int
        )
        .reindex(
            columns=
                train_encoded_features.columns,
            fill_value=0
        )
    )

    train_encoded = pd.concat(
        [
            train_meta.reset_index(
                drop=True
            ),
            train_encoded_features.reset_index(
                drop=True
            )
        ],
        axis=1
    )

    test_encoded = pd.concat(
        [
            test_ids.reset_index(
                drop=True
            ),
            test_encoded_features.reset_index(
                drop=True
            )
        ],
        axis=1
    )

    return (
        train_encoded,
        test_encoded,
        categorical_cols,
        # encoding_audit
    )


In [19]:
train_encoded, test_encoded, encoded_categorical_cols = one_hot_encode_train_test(train, test)

print(
    "Categorical columns encoded:",
    len(encoded_categorical_cols)
)

print(
    "Encoded train:",
    train_encoded.shape
)

print(
    "Encoded test :",
    test_encoded.shape
)

Categorical columns encoded: 16
Encoded train: (307511, 514)
Encoded test : (48744, 512)


## Export

In [20]:
# Save final clean datasets
FINAL_TRAIN_CLEAN_PARQUET = (
    PROCESSED_DIR
    / "final_train_clean.parquet"
)

FINAL_TEST_CLEAN_PARQUET = (
    PROCESSED_DIR
    / "final_test_clean.parquet"
)

train.to_parquet(
    FINAL_TRAIN_CLEAN_PARQUET,
    index=False
)

test.to_parquet(
    FINAL_TEST_CLEAN_PARQUET,
    index=False
)


# Save final encoded datasets
FINAL_TRAIN_ENCODED_PARQUET = (
    PROCESSED_DIR
    / "final_train_encoded.parquet"
)

FINAL_TEST_ENCODED_PARQUET = (
    PROCESSED_DIR
    / "final_test_encoded.parquet"
)

train_encoded.to_parquet(
    FINAL_TRAIN_ENCODED_PARQUET,
    index=False
)

test_encoded.to_parquet(
    FINAL_TEST_ENCODED_PARQUET,
    index=False
)

print(
    "Saved:",
    FINAL_TRAIN_CLEAN_PARQUET
)
print(
    "Saved:",
    FINAL_TEST_CLEAN_PARQUET
)
print(
    "Saved:",
    FINAL_TRAIN_ENCODED_PARQUET
)
print(
    "Saved:",
    FINAL_TEST_ENCODED_PARQUET
)


Saved: ../data/processed/final_train_clean.parquet
Saved: ../data/processed/final_test_clean.parquet
Saved: ../data/processed/final_train_encoded.parquet
Saved: ../data/processed/final_test_encoded.parquet
